# Exp 05: Augmentations v2

Card: [exp05_augmentations_v2.md](../../experiments/exp05_augmentations_v2.md)

In [ ]:
import os

import albumentations as A
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from PIL import Image

from core.augment import V2
from core.config import IMG_SIZE, ROOT, SEED, TRAIN_DIR
from core.cv import VIP, CVResult, Config, CrossValidator
from core.data import read_train
from core.gradcam import GradCamFigures
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-augmentations-v2-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_SHOW = 2  # images per Grad-CAM figure
else:
    OVERRIDES = {}
    N_SHOW = 4
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp05_augmentations_v2" / "resnet18.toml", **OVERRIDES
)
CURRENT = Config.from_toml(
    CONFIG_DIR / "exp04_oversampling_vip" / "resnet18.toml"
)  # trained in exp04, only read
FIGURES = ROOT / "sections" / "model" / "figures"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode; categorical slots in fixed order
MUTED = "#898781"
SERIES = {"current (exp04)": "#eb6834", "v2": "#1baf7a"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## v2 draws

In [ ]:
N_DRAWS = 6
show = A.Compose(V2(IMG_SIZE).transforms())  # without Normalize, ToTensorV2
train_df = read_train()
fig, axes = plt.subplots(
    len(VIP), N_DRAWS + 1, figsize=(2.1 * (N_DRAWS + 1), 2.4 * len(VIP))
)
for cls, row_axes in zip(VIP, axes):
    image_id = (
        train_df[train_df[cls] == 1].sample(1, random_state=SEED)["image_id"].item()
    )
    image = np.array(
        Image.open(TRAIN_DIR / f"{image_id}.jpg")
        .convert("RGB")
        .resize((IMG_SIZE, IMG_SIZE))
    )
    for i, ax in enumerate(row_axes):
        ax.imshow(image if i == 0 else show(image=image)["image"])
        ax.set_axis_off()
        if cls == VIP[0]:
            ax.set_title("Original" if i == 0 else f"Draw {i}", fontsize=9)
    row_axes[0].text(
        -0.08,
        0.5,
        cls,
        transform=row_axes[0].transAxes,
        rotation=90,
        va="center",
        ha="right",
    )
fig.tight_layout()
figures = {"exp05_augmentations_v2_draws.png": fig}
plt.show()

## Training

In [ ]:
results = {
    "current (exp04)": CVResult.load(CURRENT),
    "v2": CrossValidator(CONFIG).run(),
}

## Training curves

In [ ]:
panels = [
    ("Loss (weighted BCE)", "train_loss_epoch", "val_loss"),
    ("mAP", "train_map", "val_map"),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, train_key, val_key) in zip(axes, panels):
    for name, r in results.items():
        epochs = np.arange(1, r.config.epochs + 1)
        ax.plot(epochs, r.mean_curve(train_key), color=SERIES[name], ls=":")
        ax.plot(epochs, r.mean_curve(val_key), color=SERIES[name], label=name)
    ax.set(title=f"{title}, mean over {len(CONFIG.folds)} folds", xlabel="epoch")
axes[1].plot([], [], color=MUTED, label="validation")
axes[1].plot([], [], color=MUTED, ls=":", label="train")
axes[1].legend(loc="lower right", title="augmentations", alignment="left")
fig.tight_layout()
figures["exp05_augmentations_v2_training.png"] = fig
plt.show()

## AP comparison

In [ ]:
fold_ap_df = pd.concat(
    {name: r.fold_ap for name, r in results.items()}, names=["augmentations"]
)
ap = fold_ap_df.groupby("augmentations", sort=False).mean()
ap.round(3)

In [ ]:
# difference to exp01 counts only if it has same sign on both folds
diff = (
    results["v2"].fold_ap - results["current (exp04)"].fold_ap
).T.add_prefix("diff fold ")
diff["same sign on both folds"] = (diff > 0).all(axis=1) | (diff < 0).all(axis=1)
diff.round(3)

## Grad-CAM

In [ ]:
cams = GradCamFigures(CONFIG)
for (cls, outcome), fig in cams.figures(VIP, N_SHOW).items():
    figures[f"exp05_augmentations_v2_grad_cam_{cls.lower()}_{outcome}.png"] = fig
plt.show()

In [ ]:
with Experiment(CONFIG.experiment).run("summary", {"val_mode": "fast"}) as run:
    mlflow.log_table(fold_ap_df.reset_index(), "fold_ap.json")
    mlflow.log_table(diff.reset_index(names="metric"), "diff.json")
    mlflow.log_metrics({f"threshold_{c}": cams.thresholds[c] for c in VIP})
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)